## Best Strategies for Splitting: 

### Steps:

[Sample Example: Parent-Child Document]:

**Large Documents Handling: [Ingestion Phase - Indexing]**

1. Parent Chunks: Splitting the large textual data into logical & large chunks. [These chunks are designed for the LLM to read]. *(around 400-500 wrds)*

2. Child Chunks: Taking the chunk & split it further into smaller, more granular child chunks.
[Designed for accurate vector search]. *(around 100 wrds)*

3. Embedding & Store: To generate the embeddings of the child chunks only [To store into vector db]. Store parent chunks in separate, simple db.


**Retrieval Phase: [Answering a Question]**

1. Search: Convert the user query into vector embeddings and perform the similarity search (cosine generally used).

2. Fetch Parent: based on mapping we performed while ingestion.

3. Synthesize: To pass the large, context-rich parent chunks to the LLM along with the user's question for o/p generation.

In [11]:
import os
import json
from dotenv import load_dotenv
from langchain.document_loaders import TextLoader

In [6]:
SAMPLES_DIR = os.path.abspath('samples')
LONG_TEXT_PTH = os.path.join(SAMPLES_DIR, 'long_text.txt')

In [7]:
loader = TextLoader(
    file_path = LONG_TEXT_PTH,
    autodetect_encoding=True
)
loader

In [8]:
documents = loader.load()

In [13]:
metadata = documents[0].metadata

print(json.dumps(metadata, indent=2))

{
  "source": "e:\\00_SCULPTSOFT\\training-internship\\Training-Tasks---SculptSoft\\Gen AI - LLMs\\basics\\samples\\long_text.txt"
}


### Text Splitters:

In [14]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)

child_spiltter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 40 
)

### Embedding model:

In [27]:
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model_name = 'models/gemini-embedding-exp-03-07'
embedding_model_name = "all-MiniLM-L6-v2"

load_dotenv()

embeddings = HuggingFaceEmbeddings(
    model = embedding_model_name
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Vector DB/Store Setup:

In [28]:
# from langchain_community.vectorstores import Chroma
from langchain_chroma.vectorstores import Chroma
from langchain.storage import InMemoryStore

vector_store = Chroma(
    collection_name = "testing_retrievel",
    embedding_function = embeddings
)

store = InMemoryStore()

### Building Retriever:


In [29]:
from langchain.retrievers import ParentDocumentRetriever

retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=store,
    child_splitter=child_spiltter,
    parent_splitter=parent_splitter
)

### Adding document to the retriever along with performing the splitting, embeddings.

In [30]:
retriever.add_documents(documents, ids=None)

Checking child chunks...

In [37]:
child_docs = vector_store.similarity_search("fastest-growing major economies")

child_docs[0].page_content

'Economic Growth and Development'

In [38]:
retrieved_chunk = retriever.get_relevant_documents("fastest-growing major economies")

print(retrieved_chunk[0].page_content)

India's commitment to democratic values is evident in its regular free and fair elections. With a multi-party system and a robust electoral framework, democracy in India thrives despite its complexities. Institutions such as the judiciary, Election Commission, and Comptroller and Auditor General ensure checks and balances.

The Constitution of India, adopted in 1950, guarantees fundamental rights, including equality before the law, freedom of expression, religious freedom, and protection against discrimination. It also mandates duties for citizens, ensuring that rights are matched with responsibilities.

Economic Growth and Development
India has emerged as one of the fastest-growing major economies in the world. From a primarily agrarian economy at independence, it has diversified into industry and services. Sectors like information technology, pharmaceuticals, textiles, automobiles, space research, and telecommunications have driven India’s economic rise.


---
Future Use: This retrieved Chunk now can be passed to LLM along with the user query for accurate answers.

In [45]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

llm = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash',
    temperature = 0.7,
    max_tokens=None,
    max_retries=2
)

QA-Chain Building: 

In [ ]:
TEMPLATE = """
Answer the question based only on the following context:
{context}
Question: {question}

"""

In [51]:
from langchain.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [55]:
from langchain_core.runnables import RunnablePassthrough

chain_rag = {
    "context" :  retriever | (lambda docs: "\n\n".join(doc.page_content for doc in docs)),
    "question" : RunnablePassthrough()
} | prompt | llm | StrOutputParser()

In [56]:
chain_rag.invoke("How would you comment on Indian growing economy?")

'Based on the provided context, India has emerged as one of the fastest-growing major economies in the world. It has diversified from a primarily agrarian economy at independence into industry and services. Key sectors like information technology, pharmaceuticals, textiles, automobiles, space research, and telecommunications have driven this economic rise.'

---

By Kirtan Ghelani `@SculptSoft`